# RAG 파이프라인

In [44]:
import json, re, sqlite3
import ollama
import chromadb

In [63]:
EMBED_MODEL    = "qwen3-embedding:8b"
ANSWER_MODEL   = "qwen3:8b"
CLASSIFY_MODEL = "qwen3.5:9b"
CHROMA_PATH    = "./chroma_db"
AUDIT_TABLE_DB = "./audit_tables.db"
K = 4   # 연도별 텍스트 청크 수
P = 3   # 연도별 표 수

chroma  = chromadb.PersistentClient(path=CHROMA_PATH)
vdb1    = chroma.get_collection("vdb1_text")
vdb2    = chroma.get_collection("vdb2_table")

## 1. 질문 분류기 (그대로 가져옴)

In [53]:
CLASSIFY_PROMPT = """
당신은 삼성전자 감사보고서 QA 시스템의 질문 분류기입니다. 현재 기준 연도는 2024년입니다.

사용자 질문을 분석하여 아래 JSON만 출력하세요. 설명이나 다른 텍스트는 출력하지 마세요.

{"query_type": "general" | "numeric", "years": [정수, ...]}

## 분류 기준
- general: 감사 의견, 정책, 전략, 위험 요인 등 정성적 질문
- numeric: 매출액, 비율, 증감, 재무제표 수치 등 수치/표 중심 질문

## 연도 추출 규칙
- 언급된 연도 그대로 추출
- 최근 N년 → 2024 기준 역순으로 N개
- 작년 → [2023], 올해/최신 → [2024]
- 연도 언급 없으면 → [2024]
""".strip()


def classify(query: str) -> dict:
    response = ollama.chat(
        model=CLASSIFY_MODEL,
        messages=[
            {"role": "system", "content": CLASSIFY_PROMPT},
            {"role": "user",   "content": query},
        ],
        options={"temperature": 0.0},
        think=False
    )
    raw = re.sub(r"<think>.*?</think>", "", response["message"]["content"], flags=re.DOTALL).strip()
    result = json.loads(re.search(r"\{.*\}", raw, re.DOTALL).group())
    result["original_query"] = query
    return result

## 2. Retrieval

In [54]:
def retrieve(cls: dict) -> dict:
    query = cls["original_query"]
    years = cls["years"]
    q_emb = ollama.embed(model=EMBED_MODEL, input=[query]).embeddings

    chunks = []
    for year in years:
        res = vdb1.query(query_embeddings=q_emb, n_results=K, where={"year": year})
        for doc, meta in zip(res["documents"][0], res["metadatas"][0]):
            chunks.append({"year": year, "text": doc, "note_title": meta.get("note_title", "")})

    tables = []
    if cls["query_type"] == "numeric":
        for year in years:
            res = vdb2.query(query_embeddings=q_emb, n_results=P, where={"year": year})
            table_ids = res["ids"][0]
            conn = sqlite3.connect(AUDIT_TABLE_DB)
            cur  = conn.cursor()
            cur.execute(
                f"SELECT title, markdown, footnotes FROM tables WHERE table_id IN ({','.join('?'*len(table_ids))})",
                table_ids,
            )
            for row in cur.fetchall():
                # 제목 + 마크다운 + footnotes 병합
                merged = f"[제목] {row[0]}\n\n{row[1]}"
                if row[2]:
                    merged += f"\n\n[주석] {row[2]}"
                tables.append({"year": year, "content": merged})
            conn.close()

    return {"chunks": chunks, "tables": tables}

## 3. 컨텍스트 조립 & 답변 생성

In [81]:
ANSWER_SYSTEM = """당신은 삼성전자 감사보고서 전문 분석가입니다.

## 역할
제공된 컨텍스트만을 근거로 질문에 답하세요.
컨텍스트에 없는 내용은 '해당 정보가 없습니다'라고 답하세요.

## 컨텍스트 구성
아래 컨텍스트는 사용자 질문과의 유사도 순으로 감사보고서에서 검색된 텍스트 청크와 표들입니다.
질문에 실제로 관련 있는 내용만 선별하여 답변에 활용하고, 관련 없는 내용은 무시하세요.

## 표 해석 규칙
- 금액 단위: 별도 언급이 없는 이상 백만원 단위 (예: 1,496,684 = 1,496,684백만원)
- 당기: 해당 표가 수록된 감사보고서의 연도 (예: 2023년 감사보고서의 당기 = 2023년)
- 전기: 당기의 1년 전 (예: 2023년 감사보고서의 전기 = 2022년)
- 표의 값을 가져올때 당기 데이터인지 전기 데이터인지 확인 후 어느 값을 가져와야 하는지 명확히 판단해. 질문이 2024년이고 표의 출처가 2024년이면 당연히 당기를 가져와야겠지? 전기라는 단어가 포함된 표에서 절대 가져오면 안돼.
- 중요한점: 표의 값을 가져오는 것에 중점을 두고, 빠르게 답을 생성할 수 있도록 수치적 연산은 지양해."""


In [75]:
def build_context(retrieved: dict) -> str:
    lines = ["### 텍스트"]
    for c in retrieved["chunks"]:
        label = f"[{c['year']}년 / {c['note_title']}]" if c["note_title"] else f"[{c['year']}년]"
        lines.append(f"{label}\n{c['text']}")

    if retrieved["tables"]:
        lines.append("\n### 표")
        for t in retrieved["tables"]:
            lines.append(f"[{t['year']}년]\n{t['content']}")
    return "\n".join(lines)

def answer(query: str, context: str) -> str:
    response = ollama.chat(
        model=ANSWER_MODEL,
        messages=[
            {"role": "system", "content": ANSWER_SYSTEM},
            {"role": "user",   "content": f"{context}\n\n질문: {query}"},
        ],
        options={"temperature": 0.3},
        think=True,
    )
    return response["message"]["content"]

## 4. 전체 파이프라인

In [76]:
#단일질문 테스트용
def run(query: str):
    cls       = classify(query)
    print(f"[분류] {cls}\n")

    retrieved = retrieve(cls)
    print(f"[검색] 텍스트 {len(retrieved['chunks'])}개, 표 {len(retrieved['tables'])}개\n")

    context   = build_context(retrieved)
    response  = answer(query, context)
    print(f"[답변]\n{response}")

In [78]:
REWRITE_PROMPT = """당신은 검색 엔진을 위한 질문 재작성기(Query Rewriter)입니다.
사용자의 이전 대화 기록과 현재 질문을 바탕으로, 대화 문맥 없이도 독립적으로 이해될 수 있는 완전한 문장으로 현재 질문을 재작성하세요.

[규칙]
1. 대명사나 생략된 맥락(연도, 주어, 대상, 속성 등)을 이전 대화를 참고하여 명확하게 복원하세요.
2. 현재 질문이 이미 완전하다면 원래 질문을 그대로 출력하세요.
3. 설명, 인사말, 따옴표 등은 절대 출력하지 말고 오직 재작성된 질문 텍스트만 출력하세요."""

def rewrite_query(query: str, history: list) -> str:
    # 히스토리가 없으면 원래 질문 그대로 반환
    if not history:
        return query

    messages = [{"role": "system", "content": REWRITE_PROMPT}]

    # 컨텍스트가 너무 길어지지 않도록 최근 4개(2턴)의 대화만 참조
    for msg in history[-4:]:
        messages.append(msg)

    messages.append({"role": "user", "content": f"현재 질문: {query}"})

    response = ollama.chat(
        model=CLASSIFY_MODEL,  # 빠르고 지시 수행 능력이 좋은 분류 모델 사용
        messages=messages,
        options={"temperature": 0.0},
        think=False
    )

    # 불필요한 태그가 섞여 나올 경우를 대비한 클렌징
    rewritten = re.sub(r"<think>.*?</think>", "", response["message"]["content"], flags=re.DOTALL).strip()
    return rewritten

def answer(query: str, context: str, history: list) -> str:
    messages = [{"role": "system", "content": ANSWER_SYSTEM}]
    messages.extend(history)
    messages.append({"role": "user", "content": f"{context}\n\n질문: {query}"})

    response = ollama.chat(
        model=ANSWER_MODEL,
        messages=messages,
        options={"temperature": 0.6},
        think=True,
    )
    return response["message"]["content"]


def run_multiturn():
    history = []
    print("감사보고서 QA 시스템입니다. 종료하려면 'q'를 입력하세요.\n")

    while True:
        original_query = input("질문: ").strip()
        if original_query.lower() == "q":
            break

        # 1. 질문 재작성 수행
        standalone_query = rewrite_query(original_query, history)
        print(f"[재작성된 질문] {standalone_query}\n")

        # 2. '재작성된 질문'을 기준으로 분류 및 검색
        # (예: "2023년도 알려줘" -> "2023년의 감사의견에 대해 알려줘")
        cls       = classify(standalone_query)
        print(f"[분류] {cls}\n")

        retrieved = retrieve(cls)
        print(f"[검색] 텍스트 {len(retrieved['chunks'])}개, 표 {len(retrieved['tables'])}개\n")

        # 3. 컨텍스트 조립 및 답변 생성
        context   = build_context(retrieved)

        # 답변 생성 시에는 원래 질문(original_query)을 그대로 넣어주는 것이
        # AI가 대화의 자연스러운 흐름을 이어가는 데 유리합니다.
        response  = answer(original_query, context, history)

        # 4. 히스토리 업데이트 (재작성 전의 원본 대화 흐름을 저장)
        history.append({"role": "user",      "content": original_query})
        history.append({"role": "assistant", "content": response})

        print(f"[답변]\n{response}\n")

In [8]:
run("2023년 감사인의 의견은 무엇인가요?")

[답변]
2023년 감사인의 의견은 다음과 같습니다:  
- **재무제표 감사에 대한 의견**: 삼성전자 주식회사의 2023년 12월 31일 현재 재무상태 및 보고기간의 재무성과, 현금흐름은 **한국채택국제회계기준에 따라 중요성의 관점에서 공정하게 표시**되고 있다고 판단하였습니다.  
- **내부회계관리제도 감사에 대한 의견**: 회사의 내부회계관리제도는 **「내부회계관리제도 설계 및 운영 개념체계」에 따라 중요성의 관점에서 효과적으로 설계 및 운영**되고 있다고 판단하였습니다.  

감사인의 결론은 재무제표와 내부회계관리제도 모두에 대해 **적정의견**(적정하게 표시됨)을 표명한 것으로, 감사보고서에서 명시되었습니다.


In [82]:
run_multiturn()

KeyboardInterrupt: Interrupted by user

In [64]:
run("2023년 감사인의 의견은 무엇인가요?")

[답변]
2023년 감사인의 의견은 **"별첨된 회사의 재무제표는 회사의 2023년 12월 31일 현재의 재무상태와 동일로 종료되는 보고기간의 재무성과 및 현금흐름을 한국채택국제회계기준에 따라 중요성의 관점에서 공정하게 표시하고 있다"**는 것으로, **적정의견**(Unqualified Opinion, 청렴의견)을 표명한 것으로 확인됩니다. 이는 재무제표가 공정하게 작성되었음을 의미합니다.


In [10]:
run("Samsung Display Vietnam Co., Ltd. (SDV)의 2023년 당기 매출알려줘")

[답변]
Samsung Display Vietnam Co., Ltd. (SDV)의 2023년 당기 매출은 **1,496,684**입니다.  

이 데이터는 제공된 자료에서 "30. 특수관계자와의 거래 > 나. 당기 및 전기 중 특수관계자와의 매출ㆍ매입 등 거래 내역" 테이블에 기재된 것으로, **"매출 등"** 항목에 해당합니다. 통화는 명시되지 않았으나, 베트남 기업의 경우 일반적으로 **베트남 동**(VND)으로 표기됩니다.  

참고로, 이 수치는 삼성디스플레이와의 특수관계자 거래 내역 중 SDV와의 매출 금액입니다.


In [65]:
run("2023년의 할인율과 미래임금상승률을 알려줘.")

[답변]
2023년의 할인율은 **5.6%**이며, 미래임금상승률(인플레이션 효과 포함)은 **5.6%**입니다.


In [46]:
run("2024년도 비용구조 분석을 정량적으로 분석해서 알려줘.")

[답변]
2024년도 비용구조 분석을 정량적으로 살펴보면 다음과 같습니다:

1. **총 비용**: 2024년도 총 비용은 196,691,207백만원입니다. 이는 전년도(2023년)의 181,900,387백만원보다 증가한 수준입니다.

2. **비용 항목별 분석**:
   - **원재료 등의 사용액 및 상품 매입액 등**: 87,478,581백만원으로, 총 비용의 약 44.5%를 차지합니다. 이는 전년도의 89,350,845백만원보다 약 2,000,000백만원 감소한 수준입니다.
   - **급여**: 15,247,579백만원으로, 총 비용의 약 7.7%를 차지합니다. 이는 전년도의 13,941,832백만원보다 약 1,300,000백만원 증가한 수준입니다.
   - **퇴직급여**: 915,298백만원으로, 총 비용의 약 0.5%를 차지합니다. 이는 전년도의 736,004백만원보다 약 180,000백만원 증가한 수준입니다.
   - **감가상각비**: 30,285,298백만원으로, 총 비용의 약 15.4%를 차지합니다. 이는 전년도의 25,619,594백만원보다 약 4,600,000백만원 증가한 수준입니다.
   - **복리후생비**: 3,414,856백만원으로, 총 비용의 약 1.7%를 차지합니다. 이는 전년도의 3,307,047백만원보다 약 108,000백만원 증가한 수준입니다.
   - **유틸리티비**: 5,759,175백만원으로, 총 비용의 약 2.9%를 차지합니다. 이는 전년도의 5,051,847백만원보다 약 707,000백만원 증가한 수준입니다.
   - **외주용역비**: 4,505,891백만원으로, 총 비용의 약 2.3%를 차지합니다. 이는 전년도의 3,791,485백만원보다 약 714,000백만원 증가한 수준입니다.
   - **광고선전비**: 1,634,516백만원으로, 총 비용의 약 0.8%를 차지합니다. 이는 전년도의 1,747,480백만원보다 약 113,000백만원 감소한 수준입니다.
   - **판매촉진비**: 1,385,627백만원

In [47]:
run("2024년 삼성전자의 비용구조는 어떻게되지? 매출액과 영업비용, 영업외비용은 각각 얼마고, 영업비용과 영업외비용은 뭘로 구성되고 각각 얼마냐")

[답변]
2024년 삼성전자의 비용구조는 다음과 같습니다:

### 1. 매출액
- **2024년**: 209,052,241백만원
- **2023년**: 170,374,090백만원

### 2. 영업비용
- **2024년**: 44,629,735백만원
- **2023년**: 37,876,835백만원

#### 영업비용의 구성
- **급여**: 2,521,337백만원 (2024년), 2,541,935백만원 (2023년)
- **퇴직급여**: 163,548백만원 (2024년), 133,881백만원 (2023년)
- **지급수수료**: 3,091,188백만원 (2024년), 3,566,826백만원 (2023년)
- **감가상각비**: 477,811백만원 (2024년), 468,454백만원 (2023년)
- **무형자산상각비**: 398,113백만원 (2024년), 390,581백만원 (2023년)
- **광고선전비**: 1,634,516백만원 (2024년), 1,747,480백만원 (2023년)
- **판매촉진비**: 1,385,627백만원 (2024년), 1,298,806백만원 (2023년)
- **운반비**: 865,698백만원 (2024년), 541,902백만원 (2023년)
- **서비스비**: 1,292,125백만원 (2024년), 981,945백만원 (2023년)
- **기타판매비와관리비**: 2,641,755백만원 (2024년), 2,352,176백만원 (2023년)

### 3. 영업외비용
- **2024년**: 540,542백만원
- **2023년**: 375,723백만원

#### 영업외비용의 구성
- **기타수익 및 기타비용**: 23. 기타수익 및 기타비용의 내역은 제공되지 않았습니다.
